# Quantitative Trading Strategy Analysis

This notebook demonstrates a complete end-to-end analysis of a momentum-based trading strategy using:
- Data ingestion from Yahoo Finance
- Dual Moving Average Crossover strategy with RSI filter
- Vectorized backtesting
- Walk-forward validation
- Performance metrics and visualization

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Import our modules
from src.data_ingestion import fetch_data, fetch_multiple
from src.strategy import DualMAStrategy, MeanReversionStrategy, MomentumStrategy
from src.backtest import Backtester
from src.metrics import calculate_metrics, PerformanceReport, compare_strategies
from src.walk_forward import WalkForwardValidator

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("Setup complete!")

## 2. Data Ingestion

Fetch historical price data from Yahoo Finance.

In [ ]:
# Fetch data for analysis
ticker = 'AAPL'
start_date = '2020-01-01'
end_date = '2024-01-01'

print(f"Fetching data for {ticker}...")
data = fetch_data(ticker, start=start_date, end=end_date)

print(f"\nData shape: {data.shape}")
print(f"Date range: {data.index[0].strftime('%Y-%m-%d')} to {data.index[-1].strftime('%Y-%m-%d')}")
print("\nFirst few rows:")
data.head()

In [ ]:
# Visualize price data
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})

# Price chart
axes[0].plot(data.index, data['Close'], label='Close Price', linewidth=1.5)
axes[0].set_title(f'{ticker} Price History', fontsize=14)
axes[0].set_ylabel('Price ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Volume chart
axes[1].bar(data.index, data['Volume'], alpha=0.7, color='blue')
axes[1].set_ylabel('Volume')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Strategy Implementation

We'll implement and test a **Dual Moving Average Crossover with RSI Filter** strategy.

### Strategy Logic:
- **Entry**: Fast MA crosses above Slow MA AND RSI > 50 (momentum confirmation)
- **Exit**: Fast MA crosses below Slow MA OR RSI > 70 (overbought)
- **Position Sizing**: Volatility-based (ATR)

In [ ]:
# Initialize strategy
strategy = DualMAStrategy(
    fast_window=20,
    slow_window=50,
    rsi_period=14,
    rsi_overbought=70,
    position_size=0.1
)

# Generate signals
signals = strategy.generate_signals(data)

print("Signal Summary:")
print(f"  Total signals: {(signals['signal'] != 0).sum()}")
print(f"  Buy signals: {(signals['signal'] == 1).sum()}")
print(f"  Sell signals: {(signals['signal'] == -1).sum()}")

signals[['Close', 'fast_ma', 'slow_ma', 'rsi', 'signal', 'position']].tail(20)

In [ ]:
# Visualize strategy signals
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True, 
                        gridspec_kw={'height_ratios': [3, 1, 1]})

# Plot 1: Price with moving averages and signals
ax1 = axes[0]
ax1.plot(signals.index, signals['Close'], label='Close Price', linewidth=1.5, alpha=0.8)
ax1.plot(signals.index, signals['fast_ma'], label=f'Fast MA ({strategy.fast_window})', linewidth=1)
ax1.plot(signals.index, signals['slow_ma'], label=f'Slow MA ({strategy.slow_window})', linewidth=1)

# Mark buy/sell signals
buy_signals = signals[signals['signal'] == 1]
sell_signals = signals[signals['signal'] == -1]

ax1.scatter(buy_signals.index, buy_signals['Close'], marker='^', color='green', 
           s=100, label='Buy Signal', zorder=5)
ax1.scatter(sell_signals.index, sell_signals['Close'], marker='v', color='red', 
           s=100, label='Sell Signal', zorder=5)

ax1.set_title(f'{ticker} - Strategy Signals', fontsize=14)
ax1.set_ylabel('Price ($)')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Plot 2: RSI
ax2 = axes[1]
ax2.plot(signals.index, signals['rsi'], label='RSI', color='purple', linewidth=1)
ax2.axhline(y=70, color='r', linestyle='--', alpha=0.7, label='Overbought (70)')
ax2.axhline(y=50, color='gray', linestyle='--', alpha=0.7, label='Midpoint (50)')
ax2.axhline(y=30, color='g', linestyle='--', alpha=0.7, label='Oversold (30)')
ax2.fill_between(signals.index, 30, 70, alpha=0.1, color='gray')
ax2.set_ylabel('RSI')
ax2.set_ylim(0, 100)
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

# Plot 3: Position
ax3 = axes[2]
ax3.fill_between(signals.index, signals['position'], alpha=0.5, color='blue')
ax3.set_ylabel('Position')
ax3.set_xlabel('Date')
ax3.set_ylim(-0.1, 1.1)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Backtesting

Run the backtest and analyze performance.

In [ ]:
# Run backtest
backtest = Backtester(
    data=data,
    strategy=strategy,
    initial_capital=100000,
    commission=0.001,  # 0.1% commission
    slippage=0.0005,   # 0.05% slippage
)

results = backtest.run()

print("Backtest Complete!\n")
print(f"Final Equity: ${results['metrics']['final_equity']:,.2f}")
print(f"Total Return: {results['metrics']['total_return_pct']:.2f}%")
print(f"Number of Trades: {results['metrics']['num_trades']}")

In [ ]:
# Display detailed metrics
metrics = results['metrics']

print("=" * 60)
print("PERFORMANCE METRICS")
print("=" * 60)
print(f"\nReturns:")
print(f"  Total Return:        {metrics['total_return_pct']:>10.2f}%")
print(f"  Annualized Return:   {metrics['annualized_return_pct']:>10.2f}%")
print(f"  Volatility:          {metrics['volatility_pct']:>10.2f}%")

print(f"\nRisk Metrics:")
print(f"  Sharpe Ratio:        {metrics['sharpe_ratio']:>10.2f}")
print(f"  Sortino Ratio:       {metrics['sortino_ratio']:>10.2f}")
print(f"  Max Drawdown:        {metrics['max_drawdown_pct']:>10.2f}%")
print(f"  Calmar Ratio:        {metrics['calmar_ratio']:>10.2f}")

print(f"\nTrade Statistics:")
print(f"  Total Trades:        {metrics['num_trades']:>10}")
print(f"  Win Rate:            {metrics['win_rate_pct']:>10.1f}%")
print(f"  Avg Win:             ${metrics['avg_win']:>10.2f}")
print(f"  Avg Loss:            ${metrics['avg_loss']:>10.2f}")
print(f"  Profit Factor:       {metrics['profit_factor']:>10.2f}")
print(f"  Avg Trade Duration:  {metrics['avg_trade_duration_days']:>10.1f} days")
print("=" * 60)

In [ ]:
# Plot backtest results
fig = backtest.plot_results(figsize=(14, 10))
plt.show()

In [ ]:
# Trade log
trade_log = backtest.get_trade_log()
if not trade_log.empty:
    print(f"\nTrade Log ({len(trade_log)} trades):")
    print(trade_log.to_string())
else:
    print("No trades were executed.")

## 5. Walk-Forward Validation

Validate strategy robustness using walk-forward analysis.

In [ ]:
# Run walk-forward validation
print("Running Walk-Forward Validation...\n")

validator = WalkForwardValidator(
    data=data,
    strategy_class=DualMAStrategy,
    strategy_params={'fast_window': 20, 'slow_window': 50},
    n_splits=5,
)

wf_results = validator.run()
validator.print_summary()

In [ ]:
# Plot walk-forward results
fig = validator.plot_results(figsize=(14, 10))
plt.show()

## 6. Strategy Comparison

Compare multiple strategies on the same data.

In [ ]:
# Define strategies to compare
strategies = {
    'Dual MA + RSI': DualMAStrategy(fast_window=20, slow_window=50),
    'Mean Reversion (BB)': MeanReversionStrategy(window=20, num_std=2),
    'Momentum (ROC)': MomentumStrategy(roc_period=20, roc_threshold=5),
}

# Run backtests
all_results = {}

for name, strat in strategies.items():
    print(f"Running {name}...")
    bt = Backtester(data, strat, initial_capital=100000)
    all_results[name] = bt.run()

print("\nAll backtests complete!")

In [ ]:
# Compare strategies
comparison = compare_strategies(all_results)
print("\nStrategy Comparison:")
print(comparison.to_string(index=False))

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Equity curves
ax1 = axes[0]
for name, result in all_results.items():
    equity = result['equity_curve']
    ax1.plot(equity.index, equity, label=name, linewidth=1.5)

# Buy and hold benchmark
bh_returns = data['Close'] / data['Close'].iloc[0]
bh_equity = 100000 * bh_returns
ax1.plot(bh_equity.index, bh_equity, label='Buy & Hold', 
         linestyle='--', alpha=0.7, color='black')

ax1.set_title('Strategy Comparison - Equity Curves')
ax1.set_ylabel('Portfolio Value ($)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Metrics comparison
ax2 = axes[1]
metrics_to_plot = ['total_return_pct', 'sharpe_ratio', 'max_drawdown_pct']
x = np.arange(len(metrics_to_plot))
width = 0.25

for i, (name, result) in enumerate(all_results.items()):
    values = [result['metrics'].get(m, 0) for m in metrics_to_plot]
    ax2.bar(x + i * width, values, width, label=name)

ax2.set_title('Key Metrics Comparison')
ax2.set_xticks(x + width)
ax2.set_xticklabels(['Total Return (%)', 'Sharpe Ratio', 'Max DD (%)'])
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 7. Risk Analysis

Analyze risk characteristics of the strategy.

In [ ]:
# Get strategy returns
strategy_returns = results['returns']

# Calculate rolling metrics
from src.metrics import calculate_rolling_metrics

rolling = calculate_rolling_metrics(strategy_returns, window=63)

# Plot rolling metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Rolling return
axes[0, 0].plot(rolling.index, rolling['return'] * 100)
axes[0, 0].set_title('Rolling 3-Month Return')
axes[0, 0].set_ylabel('Return (%)')
axes[0, 0].grid(True, alpha=0.3)

# Rolling volatility
axes[0, 1].plot(rolling.index, rolling['volatility'] * 100, color='orange')
axes[0, 1].set_title('Rolling 3-Month Volatility')
axes[0, 1].set_ylabel('Volatility (%)')
axes[0, 1].grid(True, alpha=0.3)

# Rolling Sharpe
axes[1, 0].plot(rolling.index, rolling['sharpe'], color='green')
axes[1, 0].axhline(y=1, color='red', linestyle='--', alpha=0.5)
axes[1, 0].set_title('Rolling 3-Month Sharpe Ratio')
axes[1, 0].set_ylabel('Sharpe Ratio')
axes[1, 0].grid(True, alpha=0.3)

# Return distribution
axes[1, 1].hist(strategy_returns * 100, bins=50, alpha=0.7, edgecolor='black')
axes[1, 1].axvline(strategy_returns.mean() * 100, color='red', linestyle='--', 
                   label=f'Mean: {strategy_returns.mean()*100:.3f}%')
axes[1, 1].set_title('Daily Returns Distribution')
axes[1, 1].set_xlabel('Return (%)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Calculate drawdown periods
from src.metrics import calculate_drawdown_series

drawdowns = calculate_drawdown_series(results['equity_curve'])

print("Drawdown Periods:")
print(drawdowns.to_string())

if not drawdowns.empty:
    print(f"\nWorst Drawdown: {drawdowns['max_drawdown'].min()*100:.2f}%")
    print(f"Longest Duration: {drawdowns['duration_days'].max()} days")

## 8. Conclusions

### Summary

This notebook demonstrated:
1. **Data Ingestion**: Fetching clean historical data from Yahoo Finance
2. **Strategy Development**: Implementing a momentum-based trading strategy
3. **Backtesting**: Vectorized backtesting with realistic costs
4. **Walk-Forward Validation**: Demonstrating strategy robustness
5. **Risk Analysis**: Comprehensive performance and risk metrics

### Key Findings

- The Dual MA + RSI strategy showed [positive/negative] performance on the test data
- Walk-forward validation indicated [robust/overfit] behavior
- Risk-adjusted returns (Sharpe ratio) were [above/below] 1.0

### Limitations

- Past performance does not guarantee future results
- Transaction costs and slippage estimates may vary
- Strategy may perform differently in different market regimes

### Next Steps

- Test on multiple assets and time periods
- Add machine learning features
- Implement position sizing optimization
- Paper trade before live deployment